## 1. Import Packages

In [1]:
from pathlib import Path
import platform
import sys
import time

import numpy as np
import pandas as pd
import sklearn

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OrdinalEncoder
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    balanced_accuracy_score,
    roc_auc_score,
    average_precision_score,
    confusion_matrix,
)

## 2. Importing the eLCS implementation

In [2]:
# allows the notebook to run from either the project root or notebooks/.
if Path.cwd().name in {"notebooks", "notebooks-phase2"}:
    PROJECT_ROOT = Path.cwd().parent
else:
    PROJECT_ROOT = Path.cwd()

# path to eLCS package
ELCS_ROOT = PROJECT_ROOT / "external" / "scikit-eLCS-master"

# checks if the eLCS package exists
if not ELCS_ROOT.exists():
    raise FileNotFoundError(
        f"Expected the supplied eLCS package at: {ELCS_ROOT}"
    )

# adds eLCS to import path
if str(ELCS_ROOT) not in sys.path:
    sys.path.insert(0, str(ELCS_ROOT))

from skeLCS import eLCS

# confirms the paths and import are successful
print("Project root:", PROJECT_ROOT)
print("eLCS source:", ELCS_ROOT)
print("eLCS imported successfully")

Project root: c:\Users\leoca\Data-Engineering-and-AI-Project
eLCS source: c:\Users\leoca\Data-Engineering-and-AI-Project\external\scikit-eLCS-master
eLCS imported successfully


In [3]:
# checks the versions of python, NumPy, pandas, and scikit-learn
print("Python:", platform.python_version())
print("NumPy:", np.__version__)
print("pandas:", pd.__version__)
print("scikit-learn:", sklearn.__version__)

Python: 3.14.3
NumPy: 2.5.3
pandas: 3.0.6
scikit-learn: 1.9.1


## 3. Load the raw dataset

The baseline begins with the original `patient_data.csv` dataset. No feature engineering, scaling, outlier treatment, dimensionality reduction, or class-imbalance treatment is performed for the original baseline.

In [4]:
DATA_PATH = PROJECT_ROOT / "data" / "patient_data.csv"

# loads the data from the csv dataset
df_raw = pd.read_csv(DATA_PATH)

# checks the shape, rows, columns, missing values, and duplicates
print("Raw shape:", df_raw.shape)
print("Raw records:", len(df_raw))
print("Raw columns:", df_raw.shape[1])
print("Missing values:", df_raw.isna().sum().sum())
print("Duplicate records:", df_raw.duplicated().sum())

# displays the first 5 rows for sanity check
display(df_raw.head())

Raw shape: (280985, 39)
Raw records: 280985
Raw columns: 39
Missing values: 0
Duplicate records: 2284


,composite_key,age_level,gender,bmi_level,smoking,diabetes,age,age_normalized,bmi,hypertension,...,salt_intake,heart_rate,hdl,ldl,education_level,employment_status,source_dataset,disease_flags,sublabel,label
0,Young_Female_Normal_Never_No,Young,Female,Normal,Never,No,9,0.080460,19.20,0.0,...,8.685304,74.329073,65.651757,129.220447,Primary,Retired,diabetes,"0,0,0",N,Normal
1,Young_Female_Normal_Never_No,Young,Female,Normal,Never,No,3,0.011494,22.55,0.0,...,8.685304,74.329073,65.651757,129.220447,Primary,Retired,diabetes,"0,0,0",N,Normal
2,Young_Female_Normal_Never_No,Young,Female,Normal,Never,No,3,0.011494,22.89,0.0,...,8.685304,74.329073,65.651757,129.220447,Primary,Retired,diabetes,"0,0,0",N,Normal
3,Young_Female_Normal_Never_No,Young,Female,Normal,Never,No,3,0.011494,23.12,0.0,...,8.685304,74.329073,65.651757,129.220447,Primary,Retired,diabetes,"0,0,0",N,Normal
4,Young_Female_Normal_Never_No,Young,Female,Normal,Never,No,4,0.022989,19.61,0.0,...,8.685304,74.329073,65.651757,129.220447,Primary,Retired,diabetes,"0,0,0",N,Normal


## 4. Construct the target variable

The original dataset does not contain `heart_disease_target`. The binary target is reconstructed using the target definition established in Phase I. Records with the disease labels `HT`, `DI_HT`, `HT_HY`, or `DI_HT_HY` are assigned to the positive class.

In [5]:
# all labels that include heart disease
HEART_DISEASE_LABELS = [
    "HT",
    "DI_HT",
    "HT_HY",
    "DI_HT_HY",
]

# copies the raw data and creates the binary target
df_baseline = df_raw.copy()

df_baseline["heart_disease_target"] = (
    df_baseline["sublabel"]
    .isin(HEART_DISEASE_LABELS)
    .astype(int)
)

# displays the target counts and verify the sublabel mapping.
print(df_baseline["heart_disease_target"].value_counts().sort_index())

display(
    pd.crosstab(
        df_baseline["sublabel"],
        df_baseline["heart_disease_target"],
    )
)

heart_disease_target
0    275081
1      5904
Name: count, dtype: int64


heart_disease_target,0,1
sublabel,,
DI,31918,0
DI_HT,0,1398
DI_HT_HY,0,845
DI_HY,66473,0
HT,0,2595
HT_HY,0,1066
HY,69806,0
N,106884,0


## 5. Apply minimum baseline processing

This is not full preprocessing. It only prevents leakage and makes the data compatible with eLCS. No scaling, outlier handling, feature engineering, dimensionality reduction, or class-imbalance correction is applied in this baseline.

In [6]:
# the target-related columns that cannot be predictors
EXCLUDED_COLUMNS = [
    "heart_disease_target",
    "sublabel",
    "disease_flags",
    "heart_disease",
]

# separates the predictor variables from the target
X = df_baseline.drop(columns=EXCLUDED_COLUMNS)
y = df_baseline["heart_disease_target"].astype(int)

# displays the feature and target dimensions
print("Predictor shape:", X.shape)
print("Target shape:", y.shape)
print("Positive-class rate:", y.mean())

Predictor shape: (280985, 36)
Target shape: (280985,)
Positive-class rate: 0.021011797782799792


In [7]:
# confirms the raw records were kept and the target attribute is valid
assert len(X) == len(df_raw)
assert X.shape[1] == 36
assert y.isna().sum() == 0
assert set(y.unique()) == {0, 1}

print("Baseline dataset validation passed.")

Baseline dataset validation passed.


## 6. Identify categorical columns

We need a numeric NumPy feature array. Categorical columns must be converted to numbers.

In [8]:
# identifies the categorical columns that need to be converted to numerical
categorical_columns = X.select_dtypes(
    include=["object", "string", "category", "bool"]
).columns.tolist()

# displays the categorical columns
print("Categorical columns:", categorical_columns)

Categorical columns: ['composite_key', 'age_level', 'gender', 'bmi_level', 'smoking', 'diabetes', 'physical_activity', 'family_history', 'stress_level', 'low_hdl_cholesterol', 'high_ldl_cholesterol', 'high_blood_pressure', 'sugar_consumption', 'education_level', 'employment_status', 'source_dataset', 'label']


## 7. Create an 80/20 split

We need either an 80/20 split or k-fold cross-validation.

In [9]:
# select one reproducible seed for the initial baseline check.
SEED = 3

# create a shuffled and stratified 80/20 train/test split
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=SEED,
    stratify=y,
    shuffle=True,
)

# displays the partition sizes and positive-class proportions.
print("Training shape:", X_train.shape)
print("Testing shape:", X_test.shape)
print("Training positive rate:", y_train.mean())
print("Testing positive rate:", y_test.mean())

Training shape: (224788, 36)
Testing shape: (56197, 36)
Training positive rate: 0.021010908055590156
Testing positive rate: 0.021015356691638343


## 8. Converting Categories to numeric values

Fit the encoder only on the training data to prevent test-data leakage.

scikit-eLCS 1.2.4 cannot process text values directly. Therefore every categorical value must be converted into a number.

In [10]:
# creates an encoder that can handle unseen test-set categories
encoder = OrdinalEncoder(
    handle_unknown="use_encoded_value",
    unknown_value=-1,
)

# copies the test/train sets before replacing categorical columns
X_train_encoded = X_train.copy()
X_test_encoded = X_test.copy()

# learns category mappings from the training data and applies them to both partitions.
X_train_encoded[categorical_columns] = encoder.fit_transform(
    X_train[categorical_columns]
)
X_test_encoded[categorical_columns] = encoder.transform(
    X_test[categorical_columns]
)

# converts pandas objects into NumPy arrays (required by eLCS)
dataFeatures_train = X_train_encoded.to_numpy(dtype=float)
dataFeatures_test = X_test_encoded.to_numpy(dtype=float)
dataPhenotypes_train = y_train.to_numpy(dtype=int)
dataPhenotypes_test = y_test.to_numpy(dtype=int)

# keeps the feature names for later rule-population
dataHeaders = X.columns.to_numpy()

# confirms the arrays meet input requirements
print("Training features:", dataFeatures_train.shape)
print("Training target:", dataPhenotypes_train.shape)
print("Testing features:", dataFeatures_test.shape)
print("Testing target:", dataPhenotypes_test.shape)

assert np.isfinite(dataFeatures_train).all()
assert np.isfinite(dataFeatures_test).all()

Training features: (224788, 36)
Training target: (224788,)
Testing features: (56197, 36)
Testing target: (56197,)


## 9. Initialise the original eLCS baseline

Parameters are kept the same for the original baseline.
- learning_iterations = 10000: eLCS performed 10,000 learning iterations.
- N = 1000: maximum rule-population size.
- p_spec = 0.5: controls how many attributes are likely to be specified when covering rules are created.
- beta = 0.2: learning/update rate.
- chi = 0.8: crossover probability.
- mu = 0.04: mutation probability.
- selection_method = tournament: tournament selection was used by the genetic algorithm.
- theta_GA = 25: controls when the genetic algorithm can be applied.
- do_GA_subsumption = True: genetic-algorithm subsumption was enabled.
- do_correct_set_subsumption = False: correct-set subsumption was disabled.
- discrete_attribute_limit = 10: attributes with limited unique values are treated as discrete.
- match_for_missingness = False: no special missing-value matching was enabled, which was suitable because no missing values were detected.
- specified_attributes = []: no attributes were forced into every rule.
- reboot_filename = None: every model was trained from a new population rather than loading an earlier model.

In [11]:
# creates the original eLCS model using version 1.2.4 defaults.
model = eLCS(random_state=SEED)

# displays every parameter actually used by the baseline.
parameter_table = pd.DataFrame(
    [
        {"Parameter": name, "Value": value}
        for name, value in model.get_params().items()
    ]
)

display(parameter_table)

,Parameter,Value
0,N,1000
1,acc_sub,0.99
2,beta,0.2
3,chi,0.8
4,delta,0.1
5,discrete_attribute_limit,10
6,do_GA_subsumption,True
7,do_correct_set_subsumption,False
8,fitness_reduction,0.1
9,init_fit,0.01


## 10. Model Training

Record the start time and train the original eLCS model. Then display the total baseline training time. 

In [12]:
# records the start time and trains the original eLCS model.
start_time = time.perf_counter()

trainedModel = model.fit(
    dataFeatures_train,
    dataPhenotypes_train,
)

# training time in seconds
training_seconds = time.perf_counter() - start_time

# displays the total baseline training time and the number of iterations
print("Training time in seconds:", training_seconds)
print(trainedModel.learning_iterations)

Training time in seconds: 46.9635438999976
10000


## 11. Generate predictions

Using `predict()` and `predict_proba()` to make predictions and calculate probabilities.

In [13]:
# predicts the binary class for every testing set observation.
predictions = trainedModel.predict(dataFeatures_test)

# calculates eLCS voting probabilities for both target classes.
probabilities = trainedModel.predict_proba(dataFeatures_test)

# selects the probabilities corresponding to positive class 1.
positive_probabilities = probabilities[:, 1]

# converts the predictions into an integer NumPy array.
predictions = np.asarray(predictions).astype(int)

# displays the example predictions and probabilities.
print("First predictions:", predictions[:10])
print("First probabilities:", probabilities[:10])

First predictions: [0 0 0 0 0 0 0 0 0 0]
First probabilities: [[1.00000000e+00 0.00000000e+00]
 [1.00000000e+00 0.00000000e+00]
 [1.00000000e+00 0.00000000e+00]
 [9.99998316e-01 1.68354747e-06]
 [1.00000000e+00 0.00000000e+00]
 [1.00000000e+00 0.00000000e+00]
 [9.99997989e-01 2.01139573e-06]
 [1.00000000e+00 0.00000000e+00]
 [9.99989960e-01 1.00402689e-05]
 [1.00000000e+00 0.00000000e+00]]


## 12. Calculate baseline metrics

Approximately 2.1% of records are positive for heart disease, so accuracy alone is insufficient.

In [14]:
# calculates the classification metrics for the imbalanced binary target.
# the target is imbalanced because there are way more "no heart disease" cases than "heart disease" cases.
accuracy = accuracy_score(dataPhenotypes_test, predictions)

# precision measures how often a heart-disease prediction is correct.
precision = precision_score(
    dataPhenotypes_test,
    predictions,
    zero_division=0,
)

# recall measures the proportion of actual heart-disease cases correctly detected,
recall = recall_score(
    dataPhenotypes_test,
    predictions,
    zero_division=0,
)

# F1 combines precision and recall into one score.
f1 = f1_score(
    dataPhenotypes_test,
    predictions,
    zero_division=0,
)

# balanced accuracy measures accuracy for both classes separately.
balanced_accuracy = balanced_accuracy_score(
    dataPhenotypes_test,
    predictions,
)

# ROC-AUC measures how well the scores rank both classes
roc_auc = roc_auc_score(
    dataPhenotypes_test,
    positive_probabilities,
)

# PR-AUC summarises the precision-recall performance across score thresholds.
pr_auc = average_precision_score(
    dataPhenotypes_test,
    positive_probabilities,
)

# calculates the confusion matrix using class order 0, then 1.
matrix = confusion_matrix(
    dataPhenotypes_test,
    predictions,
    labels=[0, 1],
)

# places the metrics into a readable results table
baseline_results = pd.DataFrame(
    {
        "Metric": [
            "Accuracy", "Precision", "Recall", "F1", "Balanced accuracy", "ROC-AUC", "PR-AUC", "Training seconds",
        ],
        "Value": [
            accuracy, precision, recall, f1, balanced_accuracy, roc_auc, pr_auc, training_seconds,
        ],
    }
)

# display the classification metrics
display(baseline_results)

# display the confusion matrix with labels showing
display(
    pd.DataFrame(
        matrix,
        index=["Actual 0", "Actual 1"],
        columns=["Predicted 0", "Predicted 1"],
    )
)

,Metric,Value
0,Accuracy,0.978985
1,Precision,0.000000
2,Recall,0.000000
3,F1,0.000000
4,Balanced accuracy,0.500000
5,ROC-AUC,0.692494
6,PR-AUC,0.224123
7,Training seconds,46.963544


,Predicted 0,Predicted 1
Actual 0,55016,0
Actual 1,1181,0


## 13. Inspect eLCS-specific baseline information

In [15]:
# displays the final training performance and rule coverage
print(
    "Final training balanced accuracy:",
    trainedModel.get_final_accuracy(),
)
print(
    "Final training coverage:",
    trainedModel.get_final_instance_coverage(),
)


Final training balanced accuracy: 0.5
Final training coverage: 1.0


In [16]:
# collects the rule specificity and accuracy-weighted attribute statistics
specificity = trainedModel.get_final_attribute_specificity_list()
accuracy_weighted = trainedModel.get_final_attribute_accuracy_list()

# combines the attribute statistics with their feature names
attribute_results = pd.DataFrame({
    "Feature": dataHeaders,
    "Specificity count": specificity,
    "Accuracy-weighted count": accuracy_weighted,
})

# calculates the average rule accuracy while handling zero counts.
attribute_results["Average rule accuracy"] = (
    attribute_results["Accuracy-weighted count"]
    / attribute_results["Specificity count"].replace(0, np.nan)
)

# sorts the features by their frequency in the rule population.
attribute_results = (
    attribute_results
    .sort_values("Specificity count", ascending=False)
    .reset_index(drop=True)
)

display(attribute_results)

,Feature,Specificity count,Accuracy-weighted count,Average rule accuracy
0,label,460,448.511242,0.975024
1,source_dataset,456,443.875858,0.973412
2,diabetes,431,419.485574,0.973284
3,homocysteine_level,405,394.932505,0.975142
4,blood_pressure,396,385.218081,0.972773
5,diastolic_bp,383,373.337588,0.974772
6,systolic_bp,379,368.633839,0.972649
7,triglycerides,375,364.480314,0.971948
8,HbA1c_level,373,361.531588,0.969254
9,hypertension,373,361.187308,0.968331


In [17]:
# extracts the predicted class from every final macro-classifier.
rule_classes = [
    classifier.phenotype
    for classifier in trainedModel.population.popSet
]

# counts the macro-classifiers associated with each class.
classes, counts = np.unique(
    rule_classes,
    return_counts=True,
)

rules_by_class = pd.DataFrame({
    "Class": classes.astype(int),
    "Macro-classifier count": counts,
})

display(rules_by_class)

,Class,Macro-classifier count
0,0,797
1,1,99


## 14. Repeating the baseline 10 times

In [18]:
# 10 reproducible seeds for the repeated experiments.
SEEDS = [3, 6, 9, 12, 15, 18, 21, 24, 27, 30]

In [19]:

# creates a table recording every seed used in the repeated experiments.
seed_table = pd.DataFrame({
    "Run": range(1, len(SEEDS) + 1),
    "Seed": SEEDS,
})

display(seed_table)

,Run,Seed
0,1,3
1,2,6
2,3,9
3,4,12
4,5,15
5,6,18
6,7,21
7,8,24
8,9,27
9,10,30


In [20]:
# runs the complete original eLCS baseline using a seed
def run_baseline(seed):

    # creates a stratified 80/20 train-test split.
    X_train, X_test, y_train, y_test = train_test_split(
        X,
        y,
        test_size=0.20,
        random_state=seed,
        stratify=y,
        shuffle=True,
    )

    # creates the training-based categorical encoder.
    encoder = OrdinalEncoder(
        handle_unknown="use_encoded_value",
        unknown_value=-1,
    )

    # copies the partitions before encoding categorical columns.
    X_train_encoded = X_train.copy()
    X_test_encoded = X_test.copy()

    # learns mappings from training data and apply them to both partitions.
    X_train_encoded[categorical_columns] = encoder.fit_transform(
        X_train[categorical_columns]
    )
    X_test_encoded[categorical_columns] = encoder.transform(
        X_test[categorical_columns]
    )

    # converts the features and targets into numeric NumPy arrays.
    train_features = X_train_encoded.to_numpy(dtype=float)
    test_features = X_test_encoded.to_numpy(dtype=float)
    train_target = y_train.to_numpy(dtype=int)
    test_target = y_test.to_numpy(dtype=int)

    # initialises the original eLCS defaults using the current seed.
    model = eLCS(random_state=seed)

    # trains the model and measure its training time.
    start_time = time.perf_counter()
    model.fit(train_features, train_target)
    training_seconds = time.perf_counter() - start_time

    # generates class predictions and positive-class voting scores.
    predictions = np.asarray(
        model.predict(test_features)
    ).astype(int)

    probabilities = model.predict_proba(
        test_features
    )[:, 1]

    # extracts the four confusion-matrix values.
    tn, fp, fn, tp = confusion_matrix(
        test_target,
        predictions,
        labels=[0, 1],
    ).ravel()

    # returns the metrics and confusion-matrix results.
    return {
        "Seed": seed,
        "Accuracy": accuracy_score(
            test_target,
            predictions,
        ),
        "Precision": precision_score(
            test_target,
            predictions,
            zero_division=0,
        ),
        "Recall": recall_score(
            test_target,
            predictions,
            zero_division=0,
        ),
        "F1": f1_score(
            test_target,
            predictions,
            zero_division=0,
        ),
        "Balanced accuracy": balanced_accuracy_score(
            test_target,
            predictions,
        ),
        "ROC-AUC": roc_auc_score(
            test_target,
            probabilities,
        ),
        "PR-AUC": average_precision_score(
            test_target,
            probabilities,
        ),
        "True negatives": tn,
        "False positives": fp,
        "False negatives": fn,
        "True positives": tp,
        "Positive predictions": int(
            (predictions == 1).sum()
        ),
        "Training seconds": training_seconds,
    }

In [21]:
# stores the result from each seed.
baseline_runs = []

# run the unchanged baseline once for every seed.
for seed in SEEDS:
    print(f"Starting seed {seed}..")

    result = run_baseline(seed)
    baseline_runs.append(result)

    # displays performance and the number of positive predictions after each run.
    print(
        f"Completed seed {seed}: "
        f"balanced accuracy={result['Balanced accuracy']:.4f}, "
        f"ROC-AUC={result['ROC-AUC']:.4f}, "
        f"PR-AUC={result['PR-AUC']:.4f}, "
        f"positive predictions={result['Positive predictions']}"
    )

# combines the 10 results into one table
baseline_runs_df = pd.DataFrame(baseline_runs)

display(baseline_runs_df)

Starting seed 3..
Completed seed 3: balanced accuracy=0.5000, ROC-AUC=0.6925, PR-AUC=0.2241, positive predictions=0
Starting seed 6..
Completed seed 6: balanced accuracy=0.5000, ROC-AUC=0.7354, PR-AUC=0.1711, positive predictions=0
Starting seed 9..
Completed seed 9: balanced accuracy=0.5000, ROC-AUC=0.6086, PR-AUC=0.0782, positive predictions=0
Starting seed 12..
Completed seed 12: balanced accuracy=0.5000, ROC-AUC=0.6885, PR-AUC=0.1787, positive predictions=0
Starting seed 15..
Completed seed 15: balanced accuracy=0.5000, ROC-AUC=0.7252, PR-AUC=0.2188, positive predictions=0
Starting seed 18..
Completed seed 18: balanced accuracy=0.5000, ROC-AUC=0.6970, PR-AUC=0.2439, positive predictions=0
Starting seed 21..
Completed seed 21: balanced accuracy=0.5000, ROC-AUC=0.6830, PR-AUC=0.1565, positive predictions=0
Starting seed 24..
Completed seed 24: balanced accuracy=0.5000, ROC-AUC=0.7186, PR-AUC=0.1668, positive predictions=0
Starting seed 27..
Completed seed 27: balanced accuracy=0.5000

,Seed,Accuracy,Precision,Recall,F1,Balanced accuracy,ROC-AUC,PR-AUC,True negatives,False positives,False negatives,True positives,Positive predictions,Training seconds
0,3,0.978985,0.0,0.0,0.0,0.5,0.692494,0.224123,55016,0,1181,0,0,40.060912
1,6,0.978985,0.0,0.0,0.0,0.5,0.735379,0.171094,55016,0,1181,0,0,38.489231
2,9,0.978985,0.0,0.0,0.0,0.5,0.608623,0.078235,55016,0,1181,0,0,38.242998
3,12,0.978985,0.0,0.0,0.0,0.5,0.688478,0.178703,55016,0,1181,0,0,39.580706
4,15,0.978985,0.0,0.0,0.0,0.5,0.725154,0.218781,55016,0,1181,0,0,38.066081
5,18,0.978985,0.0,0.0,0.0,0.5,0.697034,0.243927,55016,0,1181,0,0,38.577993
6,21,0.978985,0.0,0.0,0.0,0.5,0.682991,0.156480,55016,0,1181,0,0,38.362348
7,24,0.978985,0.0,0.0,0.0,0.5,0.718644,0.166812,55016,0,1181,0,0,38.040340
8,27,0.978985,0.0,0.0,0.0,0.5,0.656257,0.202376,55016,0,1181,0,0,39.671077
9,30,0.978985,0.0,0.0,0.0,0.5,0.662989,0.184525,55016,0,1181,0,0,38.671079


### Validating the repeated runs

In [22]:
# confirm that all ten distinct-seed experiments were completed.
assert len(baseline_runs_df) == len(SEEDS)
assert baseline_runs_df["Seed"].nunique() == len(SEEDS)

# confirm that every confusion matrix accounts for the complete test set.
confusion_totals = (
    baseline_runs_df["True negatives"]
    + baseline_runs_df["False positives"]
    + baseline_runs_df["False negatives"]
    + baseline_runs_df["True positives"]
)

assert confusion_totals.nunique() == 1
assert confusion_totals.iloc[0] == len(y_test)

# confirm that the principal result columns contain no missing values.
required_result_columns = [
    "Accuracy",
    "Precision",
    "Recall",
    "F1",
    "Balanced accuracy",
    "ROC-AUC",
    "PR-AUC",
    "Training seconds",
]

assert not baseline_runs_df[
    required_result_columns
].isna().any().any()

print("All ten baseline runs passed validation.")

All ten baseline runs passed validation.


### Calculating summary statistics

In [23]:
# selects the measurements to summarise across the ten runs.
summary_columns = [
    "Accuracy",
    "Precision",
    "Recall",
    "F1",
    "Balanced accuracy",
    "ROC-AUC",
    "PR-AUC",
    "Positive predictions",
    "Training seconds",
]

# calculates the mean, standard deviation, minimum, and maximum.
baseline_summary = (
    baseline_runs_df[summary_columns]
    .agg(["mean", "std", "min", "max"])
    .transpose()
    .reset_index()
    .rename(columns={"index": "Metric"})
)

display(baseline_summary)

,Metric,mean,std,min,max
0,Accuracy,0.978985,1.170278e-16,0.978985,0.978985
1,Precision,0.000000,0.000000e+00,0.000000,0.000000
2,Recall,0.000000,0.000000e+00,0.000000,0.000000
3,F1,0.000000,0.000000e+00,0.000000,0.000000
4,Balanced accuracy,0.500000,0.000000e+00,0.500000,0.500000
5,ROC-AUC,0.686804,3.743284e-02,0.608623,0.735379
6,PR-AUC,0.182506,4.613999e-02,0.078235,0.243927
7,Positive predictions,0.000000,0.000000e+00,0.000000,0.000000
8,Training seconds,38.776276,7.251470e-01,38.040340,40.060912


### Baseline interpretation

Across the ten repeated experiments, the original eLCS baseline predicted every test observation as class 0. Consequently, accuracy remained high because approximately 97.9% of the dataset belonged to the negative class, while precision, recall, and F1-score for the positive class were all zero. Balanced accuracy remained 0.50, demonstrating that the default baseline did not provide useful discrimination at the final class-decision level. ROC-AUC and PR-AUC varied between runs because the underlying rule-voting scores changed with the random seed, despite no positive final predictions being produced.

### Summarise confusion-matrix counts

In [24]:
# summarises the confusion-matrix counts across repeated runs.
confusion_summary = (
    baseline_runs_df[
        [
            "True negatives",
            "False positives",
            "False negatives",
            "True positives",
        ]
    ]
    .agg(["mean", "std", "min", "max"])
    .transpose()
    .reset_index()
    .rename(columns={"index": "Confusion category"})
)

display(confusion_summary)

,Confusion category,mean,std,min,max
0,True negatives,55016.0,0.0,55016.0,55016.0
1,False positives,0.0,0.0,0.0,0.0
2,False negatives,1181.0,0.0,1181.0,1181.0
3,True positives,0.0,0.0,0.0,0.0


### Reproducibility Evidence

In [25]:
# creates the output directory for the original eLCS baseline.
OUTPUT_DIRECTORY = (
    PROJECT_ROOT
    / "outputs"
    / "original_elcs_baseline"
)

OUTPUT_DIRECTORY.mkdir(
    parents=True,
    exist_ok=True,
)

# saves the individual results, summary statistics, and parameters.
baseline_runs_df.to_csv(
    OUTPUT_DIRECTORY / "baseline_runs.csv",
    index=False,
)

baseline_summary.to_csv(
    OUTPUT_DIRECTORY / "baseline_summary.csv",
    index=False,
)

confusion_summary.to_csv(
    OUTPUT_DIRECTORY / "confusion_summary.csv",
    index=False,
)

parameter_table.to_csv(
    OUTPUT_DIRECTORY / "baseline_parameters.csv",
    index=False,
)

attribute_results.to_csv(
    OUTPUT_DIRECTORY / "baseline_attribute_diagnostics.csv",
    index=False,
)

rules_by_class.to_csv(
    OUTPUT_DIRECTORY / "baseline_rules_by_class.csv",
    index=False,
)
seed_table.to_csv(
    OUTPUT_DIRECTORY / "baseline_seeds.csv",
    index=False,
)

print("Baseline evidence saved to:")
print(OUTPUT_DIRECTORY.resolve())

Baseline evidence saved to:
C:\Users\leoca\Data-Engineering-and-AI-Project\outputs\original_elcs_baseline
